# Desafio 1

In [5]:
import boto3
import os
from datetime import datetime
from botocore.exceptions import ClientError

# ===========================================
# CONFIGURACIÓN - EDITA SOLO ESTAS LÍNEAS
# ===========================================
LOCAL_DIR = r'C:\Users\ingfe\Desktop\prueba_backup'
BUCKET_NAME = 'bucket-lfrf'
REGION = 'us-west-2'  # US West (Oregon) donde está tu bucket
S3_PREFIX = 'backup/'

# Crear carpeta de prueba si no existe
os.makedirs(LOCAL_DIR, exist_ok=True)
with open(os.path.join(LOCAL_DIR, 'archivo1.txt'), 'w') as f:
    f.write('Backup de prueba 1 - 18/03/2026')
with open(os.path.join(LOCAL_DIR, 'archivo2.txt'), 'w') as f:
    f.write('Backup de prueba 2 - 18/03/2026')
print(f"✅ Carpeta: {LOCAL_DIR}")
print("✅ Creados: archivo1.txt, archivo2.txt")

# ===========================================
# FUNCIÓN DE BACKUP
# ===========================================
def backup_to_s3():
    s3_client = boto3.client('s3', region_name=REGION)
    
    # Verificar bucket
    try:
        s3_client.head_bucket(Bucket=BUCKET_NAME)
        print(f"✅ Bucket '{BUCKET_NAME}' OK")
    except ClientError as e:
        if e.response['Error']['Code'] == '404':
            print(f"❌ Bucket '{BUCKET_NAME}' no encontrado")
            return
        raise
    
    hoy = datetime.now().strftime('%Y-%m-%d')
    fecha_prefix = f"{S3_PREFIX}{hoy}/"
    
    print(f"🚀 Backup → s3://{BUCKET_NAME}/{fecha_prefix}")
    
    archivos_subidos = 0
    for root, dirs, files in os.walk(LOCAL_DIR):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, LOCAL_DIR).replace(os.sep, '/')
            s3_key = f"{fecha_prefix}{relative_path}"
            
            try:
                s3_client.upload_file(local_path, BUCKET_NAME, s3_key)
                print(f"✅ {local_path} → {s3_key}")
                archivos_subidos += 1
            except Exception as e:
                print(f"❌ Error {local_path}: {e}")
    
    print(f"🎉 COMPLETADO: {archivos_subidos} archivos subidos")

# EJECUTAR
backup_to_s3()


✅ Carpeta: C:\Users\ingfe\Desktop\prueba_backup
✅ Creados: archivo1.txt, archivo2.txt
✅ Bucket 'bucket-lfrf' OK
🚀 Backup → s3://bucket-lfrf/backup/2026-03-18/
✅ C:\Users\ingfe\Desktop\prueba_backup/archivo1.txt → backup/2026-03-18/archivo1.txt
✅ C:\Users\ingfe\Desktop\prueba_backup/archivo2.txt → backup/2026-03-18/archivo2.txt
🎉 COMPLETADO: 2 archivos subidos
